In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PCA
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# 01 - data fusion

In [ ]:
# data input

# clinical record
demographic = pd.read_csv('https://github.com/FairzanIsLearning/Chemometric-learn/blob/main/Mixed/HS_Model/Raw_data_train/demographics.csv?raw=true')
patient_assignment = pd.read_csv('https://github.com/FairzanIsLearning/Chemometric-learn/blob/main/Mixed/HS_Model/Raw_data_train/patient_assignment.csv?raw=true')
demographic = pd.read_csv('https://github.com/FairzanIsLearning/Chemometric-learn/blob/main/Mixed/HS_Model/Raw_data_train/demographics.csv?raw=true')
visits1 = pd.read_csv('https://github.com/FairzanIsLearning/Chemometric-learn/blob/main/Mixed/HS_Model/Raw_data_train/visits1.csv?raw=true')
visits2 = pd.read_csv('https://github.com/FairzanIsLearning/Chemometric-learn/blob/main/Mixed/HS_Model/Raw_data_train/visits2.csv?raw=true')
treatments = pd.read_csv('https://github.com/FairzanIsLearning/Chemometric-learn/blob/main/Mixed/HS_Model/Raw_data_train/treatments.csv?raw=true')
surgeries = pd.read_csv('https://github.com/FairzanIsLearning/Chemometric-learn/blob/main/Mixed/HS_Model/Raw_data_train/surgeries.csv?raw=true')

# transcriptomics
rnaseq_embeddings = pd.read_csv('https://github.com/FairzanIsLearning/Chemometric-learn/blob/main/Mixed/HS_Model/Raw_data_train/rnaseq_embeddings.csv?raw=true')
rnaseq_metadata = pd.read_csv('https://github.com/FairzanIsLearning/Chemometric-learn/blob/main/Mixed/HS_Model/Raw_data_train/rnaseq_metadata.csv?raw=true')

# imaging
images_embeddings = pd.read_csv('https://github.com/FairzanIsLearning/Chemometric-learn/blob/main/Mixed/HS_Model/Raw_data_train/images_embeddings.csv?raw=true')
images_metadata = pd.read_csv('https://github.com/FairzanIsLearning/Chemometric-learn/blob/main/Mixed/HS_Model/Raw_data_train/images_metadata.csv?raw=true')

# target for training
targets_training = pd.read_csv('https://github.com/FairzanIsLearning/Chemometric-learn/blob/main/Mixed/HS_Model/Raw_data_train/targets.csv?raw=true')

In [3]:
summary = pd.DataFrame({
    "rows":     [len(demographic), len(patient_assignment), len(surgeries), len(treatments)],
    "patients": [demographic.patient_id.nunique(), patient_assignment.patient_id.nunique(),
                 surgeries.patient_id.nunique(), treatments.patient_id.nunique()],
}, index=["demographics", "patient_assignment", "surgeries", "treatments"])
summary["share_of_cohort"] = summary.patients / demographic.patient_id.nunique()

summary

,rows,patients,share_of_cohort
demographics,21000,21000,1.000000
patient_assignment,21000,21000,1.000000
surgeries,60036,12326,0.586952
treatments,203812,20294,0.966381


In [4]:
# data fusion 

# clinical record
central_clinical_complete = demographic.merge(patient_assignment, on=['patient_id'], how='outer') # full join
central_clinical_complete = central_clinical_complete.merge(treatments, on=['patient_id'], how='outer')
central_clinical_complete = central_clinical_complete.merge(surgeries, on=['patient_id'], how='outer')
central_clinical_complete = central_clinical_complete.merge(targets_training, on=['patient_id'], how='outer')

# visits
visit_joined = pd.concat([visits1, visits2], ignore_index=True)
# central_clinical_complete = central_clinical_complete.merge(visit_joined, on=['patient_id'], how='outer')
# merging the visits would crash colab wk
central_visit = demographic.merge(visit_joined, on=['patient_id'], how='outer')

# transcriptomics
central_transcriptomics = rnaseq_embeddings.merge(rnaseq_metadata, on=['rnaseq_sample_id'], how='outer')
central_transcriptomics = central_transcriptomics.merge(demographic, on=['patient_id'], how='inner')

# imaging
central_images = images_embeddings.merge(images_metadata, on=['image_id'], how='outer')
central_images = central_images.merge(demographic, on=['patient_id'], how='inner')

In [11]:
# data check

# duplicates
print("Duplicate treatment ids in original file:", treatments.treatment_id.duplicated().sum())
print("Duplicate surgeries ids in original file:", surgeries.surgery_id.duplicated().sum())
print("Duplicate treatment ids in central file:", central_clinical_complete.treatment_id.duplicated().sum())
print("Duplicate surgeries ids in central file:", central_clinical_complete.surgery_id.duplicated().sum())

# missing dates
onset_missing = demographic.symptom_onset_date.isna(); delay_missing = demographic.diagnostic_delay_years.isna()

print(pd.crosstab(onset_missing.rename("onset_missing"), delay_missing.rename("delay_missing"), margins=True))
rec = delay_missing & ~onset_missing        # delay can be recomputed from dates
rec2 = onset_missing & ~delay_missing       # onset can be recomputed from delay
print(f"Recoverable delay: {rec.sum()} | recoverable onset: {rec2.sum()} | truly unknown both: {(onset_missing & delay_missing).sum()}")

Duplicate treatment ids in original file: 0
Duplicate surgeries ids in original file: 0
Duplicate treatment ids in central file: 851161
Duplicate surgeries ids in central file: 994937
delay_missing  False  True    All
onset_missing                    
False          12225  3526  15751
True            3460  1789   5249
All            15685  5315  21000
Recoverable delay: 3526 | recoverable onset: 3460 | truly unknown both: 1789


# 02 -- data exploration